In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth agno lancedb sentence-transformers pillow pandas tqdm
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.0
!pip install --no-deps trl==0.22.2

In [2]:
!pip install agno lancedb sentence-transformers pillow pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.2/39.2 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 20.4 MB/s eta 0:00:00


In [3]:
!pip install nest_asyncio

In [4]:
import nest_asyncio
nest_asyncio.apply()  # ✅ MUST be before importing agno


In [5]:
from unsloth import FastVisionModel
from agno.agent import Agent
from agno.db.sqlite import SqliteDb
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.embedder.sentence_transformer import SentenceTransformerEmbedder
from agno.models.openai import OpenAIChat
from agno.vectordb.lancedb import LanceDb
from agno.tools.knowledge import KnowledgeTools
import torch
import pandas as pd
import os
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path
import json
import base64
from io import BytesIO

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
ERROR 12-05 14:10:20 [fa_utils.py:72] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
political_kb_content = """# Political Knowledge Base for Bangladesh Meme Classification


## PERSON: Sheikh Hasina
ALIASES: hasina, sheikh hasina, শেখ হাসিনা, shaikh hasina, shekh hasina, sheikh hasina, pm hasina, prime minister hasina, hasina wazed, awami leader hasina, হাসিনা, হাছিনা, সেখ হাসিনা, শাইখ হাসিনা
TYPE: person
PARTY: Awami League
DESCRIPTION:
Sheikh Hasina Wazed is the former Prime Minister of Bangladesh who served from 2009 to 2024. She is the leader of the Awami League party, whose symbol is a boat (nouka). She is the daughter of Sheikh Mujibur Rahman, the founder of Bangladesh. Party colors: red and green.


## PERSON: Khaleda Zia
ALIASES: khaleda, khaleda zia, খালেদা জিয়া, খালেদা, khalida, khalida zia, begum khaleda zia, bnp leader, বেগম খালেদা, খালেদা ঝিয়া, খালিদা জিয়া, begum zia
TYPE: person
PARTY: BNP
DESCRIPTION:
Begum Khaleda Zia is the former Prime Minister of Bangladesh and current chairperson of the Bangladesh Nationalist Party (BNP). She served as PM from 1991-1996 and 2001-2006. The BNP symbol is a sheaf of paddy (rice). She is the widow of President Ziaur Rahman.


## PERSON: Tarique Rahman
ALIASES: tarique, zarek tia,tarique rahman, তারেক রহমান, তারেক, তারিক রহমান, tareq, tareq rahman, tarek, tarek rahman, bnp acting chairman, তারিক, তারেক রাহমান
TYPE: person
PARTY: BNP
DESCRIPTION:
Tarique Rahman (born November 20, 1965/1967) is the acting chairman of the Bangladesh Nationalist Party (BNP) and son of former President Ziaur Rahman and former Prime Minister Khaleda Zia. He has been leading the party from exile in London since 2018.


## PERSON: Nahid Islam
ALIASES: nahid, nahid islam, নাহিদ ইসলাম, নাহিদ, নাহীদ ইসলাম, নাহিদ ইছলাম, quota protest leader, student leader nahid, ncp nahid
TYPE: person
PARTY: NCP
DESCRIPTION:
Nahid Islam (born 1998) is the convenor of the National Citizen(s) Party (NCP). He was the coordinator of the 2024 student movement against quotas that evolved into the campaign to oust Prime Minister Sheikh Hasina.


## PERSON: Obaidul Quader
ALIASES: obaidul quader, ওবায়দুল কাদের, ওবায়দুল, ওবাইদুল কাদের, quader, কাদের, কাদের, awami general secretary, obaidal quader
TYPE: person
PARTY: Awami League
DESCRIPTION:
Obaidul Quader (born February 1, 1952) is the General Secretary of Bangladesh Awami League. He served as Minister of Road Transport and Bridges from 2011–2024.


## PERSON: Nurul Haque Nur
ALIASES: nurul haque nur, নুরুল হক নূর, নুরুল, নূরুল হক নূর, নুরুল হক, vp nur, nur, নূর, ducsu vp
TYPE: person
PARTY: Independent/Student Leader
DESCRIPTION:
Nurul Haque Nur (born October 1991), commonly known as VP Nur, is a Bangladeshi activist elected Vice President of DUCSU in 2019.


## PERSON: Mamunul Haque
ALIASES: mamunul, mamunul haque, মামুনুল হক, মামুনুল, মামুনুল হাক, hefazat leader, mamunul haq
TYPE: person
PARTY: Hefazat-e-Islam
DESCRIPTION:
Mamunul Haque (born November 1973) is the Joint Secretary-General of Hefazat-e-Islam Bangladesh and a prominent Islamic scholar.


## PERSON: Rumin Farhana
ALIASES: rumin, rumin farhana, রুমিন ফারহানা, রুমিন, রুমীন ফারহানা, bnp lawmaker
TYPE: person
PARTY: BNP
DESCRIPTION:
Rumin Farhana is a BNP lawmaker and prominent member of the Bangladesh Nationalist Party.


## PERSON: Sarjis Alam
ALIASES: sarjis, sarjis alam, সারজিস আলম, সারজিস, সারজীস আলম, serjis, serjis alam, july protest leader, student coordinator
TYPE: person
PARTY: NCP
DESCRIPTION:
Sarjis Alam is a July Student Protest co-leader, NCP member, and significant political figure in the 2024 quota reform movement.


## PERSON: Hasnat Abdullah
ALIASES: hasnat, hasnat abdullah, হাসনাত আবদুল্লাহ, হাসনাত, হাছনাত আবদুল্লাহ, july protest leader, student coordinator hasnat
TYPE: person
PARTY: NCP
DESCRIPTION:
Hasnat Abdullah is a July Student Protest co-leader, NCP member, and key political figure in the 2024 uprising.


## PERSON: JD Vance
ALIASES: jd vance, vance, james david vance, us vice president vance, jd vans, j d vance
TYPE: person
PARTY: US Republican Party
DESCRIPTION:
JD Vance is the current Vice President of the United States.


## PERSON: Donald Trump
ALIASES: donald trump, trump, ডোনাল্ড ট্রাম্প, ডোনাল্ড, ট্রাম্প, president trump, us president, donald tramp
TYPE: person
PARTY: US Republican Party
DESCRIPTION:
Donald Trump is the current President of the United States (reelected November 2024, inaugurated January 2025).


## PERSON: Zohran Mamdani
ALIASES: zohran, zohran mamdani, zahran mamdani, mayor elect new york, nyc mayor
TYPE: person
PARTY: Democratic Socialists of America
DESCRIPTION:
Zohran Mamdani is the Mayor-elect of New York, a Muslim Democratic Socialist who won the election.


## PARTY: Awami League
ALIASES: awami league, আওয়ামী লীগ, আওয়ামি লীগ, আওয়ামি লিগ, আওমি লীগ, al, bal, bangladesh awami league, boat party, nouka, নৌকা, নোকা, নৌকার দল
TYPE: party
SYMBOL: Boat (nouka)
COLORS: Red and Green
DESCRIPTION:
The Awami League is a major political party in Bangladesh. Its symbol is a boat (nouka), often shown with red and green colors. The party is strongly associated with the slogan "Joy Bangla" and the 1971 Liberation War.


## PARTY: BNP
ALIASES: bnp, বিএনপি, বিনপি, bangladesh nationalist party, বাংলাদেশ জাতীয়তাবাদী দল, paddy party, dhan, ধানের শীষ, ধানের শিষ, ধান, sheaf of paddy
TYPE: party
SYMBOL: Sheaf of paddy (rice stalks)
COLORS: Green
DESCRIPTION:
The Bangladesh Nationalist Party (BNP) uses a sheaf of paddy (dhan er sheesh) as its symbol, typically on a green background. Founded by Ziaur Rahman, currently led by Khaleda Zia and Tarique Rahman.


## PARTY: Jamaat-e-Islami
ALIASES: jamaat, জামাত, জামায়াত, জামাআত, jamaat-e-islami, জামায়াতে ইসলামী, জামাত ই ইসলামী, jamaat islami, islamic party, জামাইত
TYPE: party
SYMBOL: Scales (balance)
DESCRIPTION:
Jamaat-e-Islami Bangladesh is an Islamist political party frequently referenced in political debates and protests. Often mentioned alongside BNP in opposition politics.


## PARTY: Hefazat-e-Islam
ALIASES: hefazat, হেফাজত, হেফাজতে, হিফাজত, hefazat-e-islam, হেফাজতে ইসলাম, হেফাজত ই ইসলাম, hefazat islam
TYPE: party
SYMBOL: Islamic symbols
DESCRIPTION:
Hefazat-e-Islam is an Islamist advocacy group in Bangladesh that has been involved in various political movements and demonstrations.


## PARTY: Chhatra Shibir
ALIASES: shibir, ছাত্র শিবির, শিবির, ছাত্রশিবির, ছাত্র সিবির, chhatra shibir, islami chhatra shibir, student shibir, chatra shibir
TYPE: party
SYMBOL: Islamic student organization symbol
DESCRIPTION:
Bangladesh Islami Chhatra Shibir is the student wing of Jamaat-e-Islami Bangladesh. Often associated with Islamist student politics.


## PARTY: NCP
ALIASES: ncp, এনসিপি, এন সি পি, national citizen party, নাগরিক দল, নাগরিক, জাতীয় নাগরিক দল, national citizens party, citizen party
TYPE: party
SYMBOL: Youth/protest movement logo
DESCRIPTION:
The National Citizen(s) Party (NCP) is a youth and student-led political party formally launched on February 28, 2025, by leaders of the 2024 uprising including Nahid Islam, Sarjis Alam, and Hasnat Abdullah.


## ORGANIZATION: Bangladesh Chhatra League
ALIASES: bcl, বিসিএল, বি সি এল, chhatra league, ছাত্রলীগ, ছাত্র লীগ, ছাত্র লিগ, student league, awami student wing, chatra league
TYPE: organization
PARTY: Awami League
DESCRIPTION:
Bangladesh Chhatra League (BCL) is the student wing of the Awami League. Look for boat symbol and red-green colors. Often involved in campus politics.


## ORGANIZATION: DUCSU
ALIASES: ducsu, ডাকসু, ডাক্সু, ডাকসো, dhaka university central students union, du student union, ঢাকা বিশ্ববিদ্যালয় কেন্দ্রীয় ছাত্র সংসদ
TYPE: organization
PARTY: Non-partisan (student body)
DESCRIPTION:
DUCSU (Dhaka University Central Students' Union) is a student political wing of Dhaka University representing various political affiliations.


## SLOGAN: Joy Bangla
ALIASES: joy bangla, জয় বাংলা, জয়বাংলা, জয় বাংলা, jai bangla, জৈ বাংলা, joi bangla, victory to bengal
TYPE: slogan
PARTY: Awami League
DESCRIPTION:
"Joy Bangla" means "Victory to Bengal" in English. It is the national slogan of Bangladesh and strongly associated with the Awami League party and the 1971 Liberation War. Highly political when used in memes.


## EVENT: 2024 Quota Protests
ALIASES: quota protest, কোটা আন্দোলন, কোটা আন্দোলন, কোটা প্রতিবাদ, quota movement, 2024 protests, student protest 2024, quota reform, কোটা সংস্কার, july protest, জুলাই আন্দোলন
TYPE: event
DATE: July-August 2024
DESCRIPTION:
In July-August 2024, massive student-led protests occurred in Bangladesh over the quota system in government jobs. These protests led to the resignation of Prime Minister Sheikh Hasina on August 5, 2024. Any meme referencing these events is highly political.


## EVENT: July Revolution 2024
ALIASES: july revolution, জুলাই বিপ্লব, জুলাই বিপ্লব, জুলাই বিপলব, july uprising, জুলাই অভ্যুত্থান, august 5, ৫ আগস্ট, 5 august, student revolution, ছাত্র বিপ্লব, hasina resignation, হাসিনা পদত্যাগ
TYPE: event
DATE: July-August 2024
DESCRIPTION:
The July Revolution 2024 refers to the student-driven uprising that began with quota reform protests and escalated into a nationwide anti-autocracy movement, culminating in the fall of Sheikh Hasina's government on August 5, 2024.


## CONCEPT: Elections
ALIASES: election, নির্বাচন, নির্বাচন, নিরবাচন, নির্বাচন, nirbachon, vote, ভোট,ভেট্টা, voat, voting, ভোটিং, ballot, ব্যালট, ব্যালট, বেলট, polling, ভোটকেন্দ্র, ভোট কেন্দ্র, poll, জনমত, electionday, নির্বাচন দিবস
TYPE: concept
DESCRIPTION:
General elections in Bangladesh involve voting for Members of Parliament. Key election-related terms indicate political content when present in memes about Bangladeshi politics.


## CONCEPT: Nomination
ALIASES: নামিনেশন, নমিনেশন, নমিনেসন, নোমিনেশন, nomination, মনোনয়ন, মনোনয়ন, মনোনয়োন, mononoyon, nominate, মনোনীত, মনোনিত, candidate, প্রার্থী, প্রার্থি, প্রাথী, prarthi, candidacy, প্রার্থিতা, প্রার্থীতা, nominee, মনোনয়নপত্র, nomination paper
TYPE: concept
DESCRIPTION:
Political candidates must receive party nominations to contest elections. Nomination processes and nomination papers are highly political topics often featured in political satire and memes.


## CONCEPT: Campaign
ALIASES: campaign, প্রচারণা, প্রচারনা, প্রচারণা, procharona, campaigning, নির্বাচনী প্রচার, নির্বাচনি প্রচার, election campaign, rally, সমাবেশ, সমাবেস, shomabesh, procession, মিছিল, মিছিল, মিসিল, michil, public meeting, জনসভা, জনসবা, jonshobha
TYPE: concept
DESCRIPTION:
Political campaigns involve rallies, processions, public meetings, and promotional activities. Campaign-related imagery and terminology in memes indicate political content.


## CONCEPT: Parliament
ALIASES: parliament, সংসদ, সংসদ, সঙ্গসদ, shangshad, jatiya sangsad, জাতীয় সংসদ, জাতিয় সংসদ, national parliament, mp, এমপি, এম পি, member of parliament, সংসদ সদস্য, সংসদ সদস্য, lawmaker, আইনপ্রণেতা, আইন প্রণেতা
TYPE: concept
DESCRIPTION:
The Jatiya Sangsad (National Parliament) of Bangladesh consists of 350 Members of Parliament (MPs). Parliamentary proceedings and MP-related content are inherently political.


## CONCEPT: Government
ALIASES: government, সরকার, সরকার, শরকার, shorkar, ruling party, ক্ষমতাসীন দল, খমতাসীন দল, ক্ষমতাসিন দল, khomotashin, administration, প্রশাসন, প্রসাসন, proshashon, cabinet, মন্ত্রিপরিষদ, মন্ত্রীপরিষদ, montriporishod, minister, মন্ত্রী, মন্ত্রি, montri
TYPE: concept
DESCRIPTION:
The government consists of the ruling party, cabinet ministers, and administrative apparatus. References to government policies, ministers, or administration indicate political content.


## CONCEPT: Opposition
ALIASES: opposition, বিরোধী দল, বিরোধি দল, বিরুদ্ধী দল, birodhi dol, opposition party, বিরোধী পক্ষ, বিরোধি পক্ষ, birodhi pokkho, opposition leader, বিরোধী নেতা, বিরোধি নেতা, anti-government, সরকার বিরোধী, সরকার বিরোধি
TYPE: concept
DESCRIPTION:
Opposition parties and leaders challenge the ruling government. Opposition politics, protests, and criticism are core political topics in Bangladeshi discourse.


## CONCEPT: Constituency
ALIASES: constituency, নির্বাচনী এলাকা, নির্বাচনি এলাকা, nirbachoni elaka, seat, আসন, আসন, ashon, electoral district, জেলা, জেলা, zila, upazila, উপজেলা, উপজেলা, thana, থানা, থানা
TYPE: concept
DESCRIPTION:
Bangladesh is divided into electoral constituencies (seats) for parliamentary elections. References to constituencies, seats, or electoral districts indicate political content.


## CONCEPT: Manifesto
ALIASES: manifesto, ইশতেহার, ইশতেহার, ইসতেহার, ishtahar, election manifesto, নির্বাচনী ইশতেহার, নির্বাচনি ইশতেহার, party manifesto, দলীয় ইশতেহার, দলিয় ইশতেহার, pledge, প্রতিশ্রুতি, প্রতিস্রুতি, protishruti, promise, ওয়াদা, ওয়াদা, ওয়াদা
TYPE: concept
DESCRIPTION:
Political manifestos outline party promises and policies for elections. Manifesto-related content or political promises are inherently political topics.


## CONCEPT: Political Alliance
ALIASES: alliance, জোট, জোট, জুট, jot, coalition, সহজোত, সহযোগ, shohojot, grand alliance, মহাজোট, মহা জোট, mohajot, unity, ঐক্য, ঐক্য, oikko, alliance partner, জোটসঙ্গী, জোট সঙ্গী
TYPE: concept
DESCRIPTION:
Political parties form alliances and coalitions for elections and governance. Alliance politics and inter-party relationships are key political topics.


## CONCEPT: Corruption
ALIASES: corruption, দুর্নীতি, দুর্নিতি, দুরনীতি, durniti, bribery, ঘুষ, ঘুস, ghush, embezzlement, আত্মসাৎ, আত্মসাত, attoshath, graft, দুর্নীতি, scam, কেলেঙ্কারি, কেলেংকারি, kelenkari
TYPE: concept
DESCRIPTION:
Corruption allegations and anti-corruption rhetoric are major political talking points in Bangladesh. Corruption-related memes about politicians are political content.


## CONCEPT: Democracy
ALIASES: democracy, গণতন্ত্র, গনতন্ত্র, গনতন্ত্র, gonotontro, democratic, গণতান্ত্রিক, গনতান্ত্রিক, gonotantrik, autocracy, স্বৈরতন্ত্র, স্বৈরতন্ত্র, শইরতন্ত্র, shoirotontro, dictatorship, একনায়কত্ব, একনায়কত্ব, eknaykotto
TYPE: concept
DESCRIPTION:
Democracy, democratic processes, and critiques of autocracy are fundamental political concepts. Democracy-related discourse in memes indicates political content.


## CONCEPT: Vote Rigging
ALIASES: vote rigging, ভোট কারচুপি, ভোট কারচুপি, ভোট কারচুপী, bhot karchhupi, election fraud, নির্বাচন জালিয়াতি, নির্বাচন জালিয়াতি, ballot stuffing, ব্যালট ভরা, ব্যালট ভরা, fake vote, জাল ভোট, জাল ভোট, manipulation, কারসাজি, কারসাজী
TYPE: concept
DESCRIPTION:
Allegations of vote rigging and election fraud are contentious political issues in Bangladesh. Such allegations in memes indicate highly political content.


## CONCEPT: Political Rally
ALIASES: rally, সমাবেশ, সমাবেস, সমাবেশ, shomabesh, mass gathering, জনসমাবেশ, জনসমাবেস, jonshomabesh, protest march, প্রতিবাদ মিছিল, প্রতিবাদ মিসিল, protest, প্রতিবাদ, প্রতিবাদ, protibad, demonstration, বিক্ষোভ, বিক্ষোভ, বিখোভ, bikhobh
TYPE: concept
DESCRIPTION:
Political rallies, mass gatherings, and protest marches are key forms of political expression. Rally-related imagery or references indicate political content.


## CONCEPT: Political Party Leadership
ALIASES: party leader, দলীয় নেতা, দলিয় নেতা, দলিও নেতা, dolyo neta, chairperson, চেয়ারপার্সন, চেয়ারপারসন, চেয়ার পার্সন, general secretary, মহাসচিব, মহা সচিব, mohashochib, president, সভাপতি, সবাপতি, shobhapoti, party chief, দলপতি, দলপতি
TYPE: concept
DESCRIPTION:
Party leadership positions like chairperson, general secretary, and president are political roles. References to party leadership indicate political content.


## CONCEPT: Political Ideology
ALIASES: ideology, মতাদর্শ, মতাদর্স, মতাদরশ, motoddorsho, leftist, বামপন্থী, বামপন্থি, বামপনথী, bampanthi, rightist, ডানপন্থী, ডানপন্থি, danpanthi, secularism, ধর্মনিরপেক্ষতা, ধর্মনিরপেক্ষতা, dhormonirpekhota, nationalism, জাতীয়তাবাদ, জাতিয়তাবাদ
TYPE: concept
DESCRIPTION:
Political ideologies including leftism, rightism, secularism, nationalism, and Islamism shape party platforms. Ideological references in memes are political content.


## CONCEPT: Political Violence
ALIASES: political violence, রাজনৈতিক সহিংসতা, রাজনৈতিক সহিংসতা, রাজনৈতিক সহিন্সতা, rajnoitik shohingshota, clash, সংঘর্ষ, সংঘর্ষ, সংঘর্স, shonghorsh, attack, হামলা, হামলা, hamla, bomb, বোমা, বোমা, বমা, boma, harthal, হরতাল, হরতাল
TYPE: concept
DESCRIPTION:
Political violence including clashes, attacks, and hartals (strikes) are significant political issues. Violence-related political content is highly political.


## CONCEPT: Press Freedom
ALIASES: press freedom, সংবাদ স্বাধীনতা, সংবাদ স্বাধিনতা, shongbad shadhinota, media, মিডিয়া, মিডিয়া, মিডিআ, journalism, সাংবাদিকতা, সাংবাদিকতা, shangbadikota, censorship, সেন্সরশিপ, সেন্সরশীপ, censor, নিয়ন্ত্রণ, নিয়ন্ত্রন
TYPE: concept
DESCRIPTION:
Press freedom, media independence, and censorship are political issues in Bangladesh. Media-related political commentary indicates political content.


## CONCEPT: Political Indicators
ALIASES: political, রাজনীতি, রাজনিতি, রাজনীতি, rajniti, political content, political meme, politics, রাজনৈতিক, রাজনৈতিক, rajnoitik
TYPE: concept
DESCRIPTION:
Political content includes: Party symbols (boat, sheaf of paddy), Political leaders (Sheikh Hasina, Khaleda Zia, Tarique Rahman, etc.), Slogans like "Joy Bangla", Government policies, Student political organizations (BCL, Shibir), Party colors (red-green for AL, green for BNP), References to elections, protests, or political events, Election jargon (vote, nomination, campaign, manifesto), Government institutions (Parliament, cabinet, ministries), Opposition politics and criticism, Corruption allegations, Democracy and autocracy debates, Political violence and rallies.
"""


# Save knowledge base to file
with open('political_knowledge.md', 'w', encoding='utf-8') as f:
    f.write(political_kb_content)


print("✅ Structured political knowledge base with spelling variations created: political_knowledge.md")


✅ Structured political knowledge base with spelling variations created: political_knowledge.md


In [7]:
print("Loading vision model...")
max_seq_length = 16384
lora_rank = 16

base_model, base_tokenizer = FastVisionModel.from_pretrained(
    model_name="arafid/Q3-8B-GRPO-Balanced-Vision-Freezed",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
    gpu_memory_utilization=0.8,
)

FastVisionModel.for_inference(base_model)
print("✅ Vision model loaded successfully")


Loading vision model...
==((====))==  Unsloth 2025.11.6: Fast Qwen3_Vl patching. Transformers: 4.57.0. vLLM: 0.12.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Vision model loaded successfully


In [8]:
import re

def load_simple_kb(path: str):
    """
    Parse a markdown KB file into a list of entries.
    Each entry: {
        "kind": "PERSON"/"PARTY"/"SLOGAN"/"EVENT",
        "name": "Sheikh Hasina",
        "aliases": ["hasina", "sheikh hasina", ...],
        "description": "..."
    }
    """
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    lines = text.splitlines()
    entries = []
    current = None
    collecting_desc = False
    desc_lines = []

    for line in lines:
        # New section
        if line.startswith("## "):
            # Flush previous
            if current is not None:
                current["description"] = "\n".join(desc_lines).strip()
                entries.append(current)
            desc_lines = []
            collecting_desc = False

            # Parse header: "## TYPE: Name"
            header = line[3:].strip()
            m = re.match(r"(\w+)\s*:\s*(.+)", header)
            if not m:
                continue
            kind, name = m.group(1).upper(), m.group(2).strip()
            current = {
                "kind": kind,
                "name": name,
                "aliases": [],
                "description": "",
            }

        elif current is not None and line.startswith("ALIASES:"):
            aliases_part = line[len("ALIASES:"):].strip()
            aliases = [a.strip().lower() for a in aliases_part.split(",") if a.strip()]
            current["aliases"] = aliases

        elif current is not None and line.startswith("DESCRIPTION:"):
            collecting_desc = True
            # Content may be on same line after colon
            after = line[len("DESCRIPTION:"):].lstrip()
            if after:
                desc_lines.append(after)

        elif current is not None and collecting_desc:
            desc_lines.append(line)

    # Flush last section
    if current is not None:
        current["description"] = "\n".join(desc_lines).strip()
        entries.append(current)

    return entries


# Simple matcher: find all entities whose alias or name appears in text
def get_kb_context_from_text(extracted_text: str, kb_entries, max_items: int = 10):
    text_lower = extracted_text.lower()
    matched = []

    for entry in kb_entries:
        name_hit = entry["name"].lower() in text_lower
        alias_hit = any(alias in text_lower for alias in entry["aliases"])
        if name_hit or alias_hit:
            matched.append(entry)

    # If nothing matched, return generic hints (optional)
    if not matched:
        return ""

    # Build a brief context block
    ctx_lines = []
    for e in matched[:max_items]:
        ctx_lines.append(f"{e['kind'].title()}: {e['name']}")
        if e["description"]:
            ctx_lines.append(f"{e['description']}")
        ctx_lines.append("")  # blank line

    return "\n".join(ctx_lines).strip()


In [9]:
simple_kb = load_simple_kb("/content/political_knowledge.md")


In [7]:
!pip install vllm

INFO: pip is looking at multiple versions of model-hosting-container-standards to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 30.7 MB/s eta 0:00:00


## OCR pipeline,database knowledge is added here

In [16]:
from qwen_vl_utils import process_vision_info

def classify_image_with_knowledge(image_path: str) -> str:
    """
    Two-pass pipeline:
    1) Use the vision model itself to read text from the meme (OCR-style)
    2) Use extracted text as a query into Agno Knowledge
    3) Combine base political knowledge + OCR-based knowledge into final prompt
    4) Ask the model to output ONLY 'Political' or 'NonPolitical'
    """
    try:
        # ------------------------------------------------------------------
        # PASS 1: OCR / TEXT EXTRACTION FROM IMAGE
        # ------------------------------------------------------------------
        ocr_prompt = """You are an OCR assistant and an image describer.
Read ALL text that appears in this meme image (translate any Bengali into proper English)
and return ONLY the raw text content, without any explanation or extra words. Give a description of the image, highlighting any political party logos.
"""

        ocr_conv = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": Image.open(image_path).convert("RGB")},
                    {"type": "text", "text": ocr_prompt},
                ],
            }
        ]

        # Standard HF multimodal encoding
        ocr_text = base_tokenizer.apply_chat_template(
            ocr_conv, tokenize=False, add_generation_prompt=True
        )
        ocr_images, ocr_videos = process_vision_info(ocr_conv)

        ocr_inputs = base_tokenizer(
            text=[ocr_text],
            images=ocr_images,
            videos=ocr_videos,
            padding=True,
            return_tensors="pt",
        ).to("cuda")

        ocr_ids = base_model.generate(**ocr_inputs, max_new_tokens=128, temperature=0.1)
        ocr_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(ocr_inputs.input_ids, ocr_ids)
        ]
        ocr_response = base_tokenizer.batch_decode(
            ocr_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        extracted_text = ocr_response

        # ------------------------------------------------------------------
        # PASS 2: KNOWLEDGE RETRIEVAL FROM AGNO USING OCR TEXT
        # ------------------------------------------------------------------
        # Base political query plus extracted meme text as extra signal
        kb_context = get_kb_context_from_text(extracted_text, simple_kb, max_items=10)

        # ------------------------------------------------------------------
        # PASS 3: FINAL CLASSIFICATION PROMPT WITH OCR + KNOWLEDGE
        # ------------------------------------------------------------------
        # Reload image for safety (could reuse from OCR step if you prefer)
        image = Image.open(image_path).convert("RGB")

        print("Extracted Text: ",extracted_text)
        print("KB Context: ", kb_context)

        final_prompt = f"""Classify this meme image as ONLY ONE of: Political or NonPolitical.

TEXT EXTRACTED FROM THE MEME:
{extracted_text}

RETRIEVED POLITICAL KNOWLEDGE (from knowledge base) based on text present in the image:
{kb_context}

You are an expert in Bangladesh politics and political content classification.
Use BOTH the extracted text and the retrieved knowledge to decide.

Political means the meme's PRIMARY content is about:
- Politicians (Sheikh Hasina, Khaleda Zia, Tarique Rahman, Nahid Islam, etc.)
- Political parties (Awami League, BNP, Jamaat, NCP, etc.)
- Party symbols (boat for AL, sheaf of paddy for BNP)
- Political slogans (Joy Bangla, Bangladesh Zindabad)
- Government policies, elections, political campaigns
- Student movements (2024 quota protests, July Revolution)
- Political ideologies, movements, or protests

NonPolitical means the meme is about anything else:
- Gender, relationships, dating
- Religion without political context
- Everyday life, work, school
- Entertainment, movies, games, sports
- Animals, food, technology
- General humor without political context

IMPORTANT INSTRUCTIONS:
1. Only classify as Political if the MAIN subject is politics.
2. If politics is just mentioned briefly or as background, classify as NonPolitical.
3. Output ONLY ONE WORD, exactly: Political or NonPolitical.
4. Do NOT output explanations, reasoning, or any extra text.
"""

        final_conv = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": final_prompt},
                ],
            }
        ]

        text = base_tokenizer.apply_chat_template(
            final_conv, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(final_conv)

        inputs = base_tokenizer(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to("cuda")

        generated_ids = base_model.generate(
            **inputs,
            max_new_tokens=5,      # Short, just one word
            temperature=0.1,
        )
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        response = base_tokenizer.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        response_upper = response.upper()

        if "POLITICAL" in response_upper and "NON" not in response_upper:
            print("Political")
            return "Political"
        elif "NONPOLITICAL" in response_upper or "NON-POLITICAL" in response_upper:
            print("NonPolitical")
            return "NonPolitical"
        else:
            return "Unclear"

    except Exception as e:
        print(f"❌ Error processing {image_path}: {str(e)}")
        return "NonPolitical"


print("✅ Classification function ready (two-pass OCR + Agno Knowledge + Qwen VL)")


✅ Classification function ready (two-pass OCR + Agno Knowledge + Qwen VL)


In [13]:
import zipfile
import os
import pandas as pd

!unzip /content/Image.zip -d /content/Image
print("\n🔍 Loading test data...")

if os.path.exists("Test.csv"):
    test_df = pd.read_csv("Test.csv")
    print(f"✅ Test.csv loaded: {len(test_df)} images to classify")
else:
    raise FileNotFoundError("Test.csv not found!")

# Find image folder
image_folder ='/content/Image/Image'
for item in os.listdir('.'):
    if os.path.isdir(item) and item not in ['sample_data', '.config', 'tmp']:
        try:
            files = os.listdir(item)
            images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if len(images) > 0:
                image_folder = item
                print(f"✅ Found {len(images)} images in: {image_folder}")
                break
        except:
            continue

if image_folder is None:
    raise FileNotFoundError("No image folder found!")

# Verify images
missing_count = sum(1 for img in test_df['Image_name']
                   if not os.path.exists(os.path.join(image_folder, img)))
if missing_count == 0:
    print(f"✅ All {len(test_df)} test images verified")
else:
    print(f"⚠️ Warning: {missing_count} images not found")

Archive:  /content/Image.zip
   creating: /content/Image/Image/
  inflating: /content/Image/Image/test0001.jpg  
  inflating: /content/Image/Image/test0002.jpg  
  inflating: /content/Image/Image/test0003.jpg  
  inflating: /content/Image/Image/test0004.jpg  
  inflating: /content/Image/Image/test0005.jpg  
  inflating: /content/Image/Image/test0006.jpg  
  inflating: /content/Image/Image/test0007.jpg  
  inflating: /content/Image/Image/test0008.jpg  
  inflating: /content/Image/Image/test0009.jpg  
  inflating: /content/Image/Image/test0010.jpg  
  inflating: /content/Image/Image/test0011.jpg  
  inflating: /content/Image/Image/test0012.jpg  
  inflating: /content/Image/Image/test0013.jpg  
  inflating: /content/Image/Image/test0014.jpg  
  inflating: /content/Image/Image/test0015.jpg  
  inflating: /content/Image/Image/test0016.jpg  
  inflating: /content/Image/Image/test0017.jpg  
  inflating: /content/Image/Image/test0018.jpg  
  inflating: /content/Image/Image/test0019.jpg  
  inf

## Single Test , use this to understand if cos sim is working

In [14]:
classification = classify_image_with_knowledge('/content/Image/Image/test0035.jpg')
print(classification)

Extracted Text:  ভূমিকম্প থেকে বাঁচতে
যদি দাড়িপাল্লায় ভেট্টা একটু দিতেন..
Kog Porichorja Kendro
রগ পরিচর্যা কেন্দ্র
KB Context:  Concept: Elections
General elections in Bangladesh involve voting for Members of Parliament. Key election-related terms indicate political content when present in memes about Bangladeshi politics.
NonPolitical


In [17]:
from tqdm import tqdm
print(f"\n🚀 Starting classification of {len(test_df)} images...\n")

predictions = []
successful = 0
failed = 0

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Classifying"):
    image_name = row['Image_name']
    image_path = os.path.join(image_folder, image_name)

    if not os.path.exists(image_path):
        predictions.append({'Image_name': image_name, 'Label': 'NonPolitical'})
        failed += 1
        continue

    try:
        # Classify using Unsloth + Agno Knowledge
        classification = classify_image_with_knowledge(image_path)
        predictions.append({'Image_name': image_name, 'Label': classification})
        successful += 1
    except Exception as e:
        predictions.append({'Image_name': image_name, 'Label': 'NonPolitical'})
        failed += 1


🚀 Starting classification of 330 images...



Classifying:   0%|          | 0/330 [00:00<?, ?it/s]

Extracted Text:  *Looses Ducsus Election
- Any comment on the election?
- I'm a bid disappointed, to be honest
- Sad, ik

[Image Description: A man in a white shirt is speaking to reporters, gesturing with his right hand. He has a serious expression. Behind him, other people are visible, one holding a phone. The setting appears to be a press conference or public event. There is no visible political party logo in the image.]
KB Context:  Party: Awami League
The Awami League is a major political party in Bangladesh. Its symbol is a boat (nouka), often shown with red and green colors. The party is strongly associated with the slogan "Joy Bangla" and the 1971 Liberation War.

Organization: DUCSU
DUCSU (Dhaka University Central Students' Union) is a student political wing of Dhaka University representing various political affiliations.

Concept: Elections
General elections in Bangladesh involve voting for Members of Parliament. Key election-related terms indicate political content when pres

Classifying:   0%|          | 1/330 [00:21<1:55:13, 21.02s/it]

Political
Extracted Text:  kya matlab iqe fue
KB Context:  


...

Political
Extracted Text:  ছাত্র সংসদ নির্বাচনগুলোতে ছাত্রদলের পারফরম্যান্স
দেখার পর তারেক রহমানের সৎ প্রতিক্রিয়া:
BETA tumse na ho payega
বাংলাদেশ
জাতীয়তাবাদী ছা�
KB Context:  Person: Tarique Rahman
Tarique Rahman (born November 20, 1965/1967) is the acting chairman of the Bangladesh Nationalist Party (BNP) and son of former President Ziaur Rahman and former Prime Minister Khaleda Zia. He has been leading the party from exile in London since 2018.

Concept: Elections
General elections in Bangladesh involve voting for Members of Parliament. Key election-related terms indicate political content when present in memes about Bangladeshi politics.

Concept: Parliament
The Jatiya Sangsad (National Parliament) of Bangladesh consists of 350 Members of Parliament (MPs). Parliamentary proceedings and MP-related content are inherently political.

Concept: Political Ideology
Political ideologies including leftism, rightism, secularism, nationalism, and Islamism shape party platforms. Ideological

Classifying: 100%|█████████▉| 329/330 [1:44:42<00:22, 22.61s/it]

Political
Extracted Text:  Elias Hossain every 5 minutes-
ভাইয়া একটা গু'জব শুনাই?
BANGLA
KB Context:  


Classifying: 100%|██████████| 330/330 [1:44:58<00:00, 19.09s/it]

NonPolitical


In [18]:
# Create DataFrame from predictions
results_df = pd.DataFrame(predictions)

# Display summary
print("\n" + "="*80)
print("✅ INFERENCE COMPLETE!")
print("="*80)
print(f"\n📊 Summary:")
print(f"   ✅ Successful: {successful}")
print(f"   ❌ Failed: {failed}")
print(f"   📝 Total: {len(predictions)}")

# Show class distribution
print(f"\n📊 Classification Distribution:")
print(results_df['Label'].value_counts())
print(f"\n📊 Percentages:")
print(results_df['Label'].value_counts(normalize=True) * 100)

# Save to CSV
output_filename = 'rag_predictions.csv'
results_df.to_csv(output_filename, index=False)

print(f"\n✅ Results saved to: {output_filename}")
print(f"\n📋 First 10 predictions:")
print(results_df.head(10))
print(f"\n📋 Last 10 predictions:")
print(results_df.tail(10))


✅ INFERENCE COMPLETE!

📊 Summary:
   ✅ Successful: 330
   ❌ Failed: 0
   📝 Total: 330

📊 Classification Distribution:
Label
NonPolitical    170
Political       160
Name: count, dtype: int64

📊 Percentages:
Label
NonPolitical    51.515152
Political       48.484848
Name: proportion, dtype: float64

✅ Results saved to: rag_predictions.csv

📋 First 10 predictions:
     Image_name         Label
0  test0001.jpg     Political
1  test0002.jpg  NonPolitical
2  test0003.jpg     Political
3  test0004.jpg     Political
4  test0005.jpg  NonPolitical
5  test0006.jpg     Political
6  test0007.jpg  NonPolitical
7  test0008.jpg     Political
8  test0009.jpg  NonPolitical
9  test0010.jpg  NonPolitical

📋 Last 10 predictions:
       Image_name         Label
320  test0321.jpg  NonPolitical
321  test0322.jpg     Political
322  test0323.jpg     Political
323  test0324.jpg     Political
324  test0325.jpg  NonPolitical
325  test0326.jpg  NonPolitical
326  test0327.jpg  NonPolitical
327  test0328.jpg     Poli

## Testing with single image